# Stop words des reviews Steam

Liste générique (`stopwordsiso`) + mots vides propres au corpus, détectés par fréquence documentaire. Lecture du ClickHouse de prod par SSH (compte lecture seule `play`), résultats interrogés avec DuckDB.

In [ ]:
import subprocess
import tempfile

import duckdb
import stopwordsiso
from sklearn.feature_extraction.text import CountVectorizer

STEAM_LANGUAGE = "english"  # code langue Steam
ISO_LANGUAGE = "en"
SAMPLE_SIZE = 20_000
MIN_DF = 5
MAX_DF = 0.3  # au-delà, un mot présent dans 30 % des reviews devient candidat
KEEP = {"not", "no", "never", "good", "great", "bad"}  # porteurs de sentiment, à ne jamais retirer

VPS = "deploy@167.235.145.180"

In [ ]:
# TODO : remplacer par ATTACH '' AS ch (TYPE clickhouse) une fois clickhouse_scanner publié
# dans les extensions communautaires (filtres et colonnes poussés à ClickHouse).
# Le mot de passe de play reste dans le conteneur ; la requête passe par stdin, sans échappement.
CLICKHOUSE_CLIENT = (
    "cd ~/apps/steam-reviews-analysis && docker compose exec -T clickhouse "
    """sh -c 'clickhouse-client --user play --password "$CLICKHOUSE_PLAY_PASSWORD"'"""
)
con = duckdb.connect()


def ch(sql: str, name: str) -> duckdb.DuckDBPyRelation:
    """Exécute du SQL ClickHouse en prod et charge le résultat dans la table DuckDB `name`."""
    with tempfile.NamedTemporaryFile(suffix=".parquet") as f:
        subprocess.run(["ssh", VPS, CLICKHOUSE_CLIENT], input=f"{sql} FORMAT Parquet".encode(), stdout=f, check=True)
        con.execute(f"CREATE OR REPLACE TABLE {name} AS FROM read_parquet(?)", [f.name])
    return con.table(name)

In [13]:
# Le texte se décompresse par blocs de 8192 lignes : on tire des blocs (1/50), puis des lignes (1/50),
# sinon tous les blocs sont lus et la requête dépasse les 60 s du compte play.
ch(
    f"""
    SELECT app_id, review_text
    FROM staging.steam_review
    WHERE app_id = 730
    --PREWHERE cityHash64(_part, intDiv(_part_offset, 8192)) % 50 = 0
    --    AND cityHash64(recommendation_id) % N = 0
        AND language = '{STEAM_LANGUAGE}'
        AND review_text_length > 20
    LIMIT {SAMPLE_SIZE}
    """,
    "sample",
)
reviews = [text for (text,) in con.sql("SELECT review_text FROM sample").fetchall()]
len(reviews)

20000

In [ ]:
generic = stopwordsiso.stopwords(ISO_LANGUAGE)
# La liste anglaise contient aussi « good » et « great » : à vérifier pour chaque langue.
sorted(generic) # & KEEP)

In [14]:
# binary=True : on compte les reviews qui contiennent le mot, pas ses répétitions.
vectorizer = CountVectorizer(binary=True, min_df=MIN_DF, token_pattern=r"(?u)\b\w[\w']+\b")
matrix = vectorizer.fit_transform(reviews)

doc_freq = dict(zip(vectorizer.get_feature_names_out(), matrix.sum(axis=0).A1 / len(reviews)))
top = sorted(doc_freq.items(), key=lambda kv: kv[1], reverse=True)

for word, freq in top[:50]:
    print(f"{word:<15} {freq:6.1%}  {'générique' if word in generic else ''}")

game             53.3%  
the              35.8%  générique
and              32.1%  générique
this             29.7%  générique
is               28.5%  générique
to               28.3%  générique
of               25.8%  générique
it               25.4%  générique
you              20.1%  générique
but              17.3%  générique
for              17.3%  générique
in               17.2%  générique
good             16.4%  générique
with             16.2%  générique
play             14.4%  
cs               13.0%  générique
fun              13.0%  
that             12.0%  générique
if               11.6%  générique
like             11.6%  générique
have             11.2%  générique
best             11.0%  générique
on               10.9%  générique
not              10.4%  générique
are              10.2%  générique
counter          10.1%  
great             9.9%  générique
strike            9.6%  
get               9.5%  générique
very              9.2%  générique
so                9.2%  g

In [15]:
# Candidats propres au corpus : fréquents mais absents de la liste générique. À relire à la main.
corpus_candidates = [word for word, freq in top if freq > MAX_DF and word not in generic]
corpus_candidates

['game']

In [16]:
stop_words = (generic | set(corpus_candidates)) - KEEP
len(stop_words)

1294

## Variantes du nom d'un jeu

Noms alternatifs et localisations d'IGDB, variantes dérivées du nom Steam, abréviations trouvées dans les reviews, puis nombre de reviews du jeu qui citent chaque variante.

In [ ]:
import re
from pathlib import Path

import httpx

ROMAN = {"i": "1", "ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8", "ix": "9", "x": "10"}
ARABIC = {arabic: roman for roman, arabic in ROMAN.items()}
TOKEN_RE = r"\\b[a-z0-9]{2,8}\\b"  # littéral SQL : \b une fois lu par ClickHouse

# Chargeur minimal : `source .env` exposerait les secrets dans le shell.
env = dict(
    line.split("=", 1)
    for line in (Path.cwd().parents[1] / ".env").read_text().splitlines()
    if "=" in line and not line.lstrip().startswith("#")
)
IGDB_CLIENT_ID, IGDB_CLIENT_SECRET = env["IGDB_CLIENT_ID"].strip(), env["IGDB_CLIENT_SECRET"].strip()
igdb_token = (
    httpx.post(
        "https://id.twitch.tv/oauth2/token",
        params={"client_id": IGDB_CLIENT_ID, "client_secret": IGDB_CLIENT_SECRET, "grant_type": "client_credentials"},
    )
    .raise_for_status()
    .json()["access_token"]
)


def igdb(endpoint: str, body: str) -> list[dict]:
    headers = {"Client-ID": IGDB_CLIENT_ID, "Authorization": f"Bearer {igdb_token}"}
    return httpx.post(f"https://api.igdb.com/v4/{endpoint}", headers=headers, content=body).raise_for_status().json()


def sql_str(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"


def derived_names(name: str) -> set[str]:
    """Titre coupé à chaque sous-titre, sans article, chiffres romains et arabes interchangés."""
    names = {name} | {name[: m.start()] for m in re.finditer(r"\s*(?:[:：]|\s[-–—]\s)", name)}
    names |= {re.sub(r"(?i)^the\s+", "", n) for n in names}
    swap = lambda m: ROMAN.get(m[0].lower()) or ARABIC.get(m[0], m[0]).upper()  # noqa: E731
    names |= {re.sub(r"(?i)\b(?:[ivx]+|\d+)\b", swap, n) for n in names}
    return names


def initials(name: str) -> list[set[str]]:
    """Une initiale par mot ; un nombre compte en entier, en romain comme en arabe."""
    out = []
    for word in re.findall(r"[^\W_]+", name.lower()):
        if word in ROMAN:
            out.append({word[0], ROMAN[word]})
        elif word.isdigit():
            out.append({word, ARABIC.get(word, word)})
        else:
            out.append({word[0]})
    return out


def is_abbreviation(token: str, name: str) -> bool:
    """Le token enchaîne des initiales du nom en partant du premier mot, article facultatif (TW3, W3, CS, GTAV)."""

    def walk(rest: str, marks: list[set[str]]) -> bool:
        return not rest or any(
            rest.startswith(mark) and walk(rest[len(mark) :], marks[i + 1 :])
            for i, choices in enumerate(marks)
            for mark in choices
        )

    marks = initials(name)
    starts = [marks, marks[1:]] if re.match(r"(?i)(the|a|an)\s", name) else [marks]
    return any(
        token.startswith(mark) and len(token) > len(mark) and walk(token[len(mark) :], start[1:])
        for start in starts
        if start
        for mark in start[0]
    )

In [ ]:
def abbreviations(app_id: int, name: str, min_reviews: int, min_lift: float) -> list[tuple[str, float]]:
    """Mots des reviews formés d'initiales du nom et bien plus fréquents ici qu'ailleurs (TR, HR… sont partout)."""
    charset = re.escape("".join(sorted({c for choices in initials(name) for mark in choices for c in mark})))
    candidates = [
        (token, hits, total)
        for token, hits, total in ch(
            f"""
            SELECT token, count() AS hits, any(total) AS total
            FROM staging.steam_review
            ARRAY JOIN arrayDistinct(extractAll(lower(review_text), '{TOKEN_RE}')) AS token
            CROSS JOIN (SELECT count() AS total FROM staging.steam_review WHERE app_id = {app_id}) AS t
            WHERE app_id = {app_id} AND match(token, {sql_str(f"^[{charset}]+$")})
            GROUP BY token
            HAVING hits >= {min_reviews}
            """,
            "abbrev_candidates",
        ).fetchall()
        if is_abbreviation(token, name)
    ]
    if not candidates:
        return []
    # Fréquence ailleurs : 1 bloc de 8192 lignes sur 200 suffit et tient sous les 60 s.
    tokens = [token for token, _, _ in candidates]
    elsewhere_total, elsewhere_hits = ch(
        f"""
        SELECT count(), sumForEach(arrayMap(t -> toUInt64(has(toks, t)), [{", ".join(map(sql_str, tokens))}]))
        FROM (
            SELECT extractAll(lower(review_text), '{TOKEN_RE}') AS toks
            FROM staging.steam_review
            PREWHERE cityHash64(_part, intDiv(_part_offset, 8192)) % 200 = 0 AND app_id != {app_id}
        )
        """,
        "abbrev_elsewhere",
    ).fetchone()
    lifts = [(hits / total) / ((other + 1) / elsewhere_total) for (_, hits, total), other in zip(candidates, elsewhere_hits)]
    return [(token, lift) for token, lift in zip(tokens, lifts) if lift >= min_lift]


def name_variants(app_id: int, min_reviews: int = 20, min_lift: float = 10) -> duckdb.DuckDBPyRelation:
    steam_name = ch(f"SELECT name FROM staging.game_detail WHERE steam_app_id = {app_id}", "game").fetchone()[0]
    variants = {v: ("steam dérivé", None) for v in derived_names(steam_name)}
    variants[steam_name] = ("steam", None)

    link = ch(f"SELECT igdb_id FROM staging.igdb_game WHERE steam_app_id = {app_id}", "igdb_link").fetchone()
    if link:
        fields = "name, alternative_names.name, alternative_names.comment, game_localizations.name, game_localizations.region.name"
        game = igdb("games", f"fields {fields}; where id = {link[0]};")[0]
        for n in derived_names(game["name"]):
            variants.setdefault(n, ("igdb dérivé", None))
        for alt in game.get("alternative_names", []):
            if "executable" not in alt.get("comment", "").lower():
                variants.setdefault(alt["name"], ("igdb alternatif", alt.get("comment")))
                for n in derived_names(alt["name"]):
                    variants.setdefault(n, ("igdb dérivé", alt.get("comment")))
        for loc in game.get("game_localizations", []):
            if "name" in loc:
                variants.setdefault(loc["name"], ("igdb localisation", loc["region"]["name"]))
                for n in derived_names(loc["name"]):
                    variants.setdefault(n, ("igdb dérivé", loc["region"]["name"]))

    for token, lift in abbreviations(app_id, steam_name, min_reviews, min_lift):
        if token.upper() not in {v.upper() for v in variants}:
            variants[token.upper()] = ("reviews", f"lift {lift:.0f}")

    # Mot entier pour l'ASCII ; sous-chaîne sinon, \b ignorant les écritures non latines.
    names = list(variants)
    tests = [
        f"match(review_text, {sql_str(r'(?i)\b' + re.escape(v) + r'\b')})"
        if v.isascii()
        else f"positionCaseInsensitiveUTF8(review_text, {sql_str(v)}) > 0"
        for v in names
    ]
    counts = ch(
        f"SELECT {', '.join(f'countIf({t})' for t in tests)} FROM staging.steam_review WHERE app_id = {app_id}",
        "variant_counts",
    ).fetchone()
    con.execute("CREATE OR REPLACE TABLE variants (variant VARCHAR, source VARCHAR, detail VARCHAR, reviews BIGINT)")
    con.executemany("INSERT INTO variants VALUES (?, ?, ?, ?)", [(v, *variants[v], n) for v, n in zip(names, counts)])
    return con.sql("SELECT * FROM variants ORDER BY reviews DESC")


name_variants(292030).show(max_rows=100, max_col_width=40)